# 00 · Preprocessing: from the raw BV-BRC export to the modelling table

**Forecasting Antibiotic Resistance with AI** · Final Year Project · Hamza Afzal (models), Muhammad Ali Mirza (data and evolution), Suleman Saleh (platform)

Every model in the project starts from one cleaned table. This notebook shows **the project's own cleaning code** (`experiments/lib/data_prep.py`, Ali's cleaning, version v7), what each step does to real raw records, and then checks that the cleaned table the models were trained on has exactly the figures reported in the Data chapter (`docs/report/Chapter_Data.md`, sections 3 and 5).

| Section | What it shows |
|---|---|
| 1 | The raw records as BV-BRC exports them |
| 2–7 | Each cleaning step on 1.5 million real raw records: labels, MIC, organism, antibiotic names, provenance, one row per genome and drug |
| 8 | The whole cleaning function, run in one call |
| 9 | **The table the models were trained on: every reported figure recomputed** |
| 10 | What each model takes from that table (features), including a genome turned into model inputs |

Run time: about 2 minutes. Needs `Data/amr_full/` (raw records) and `experiments/cache/clean_v7_amr_full_norm.pkl` (the cleaned table, on the team's Google Drive, `docs/DATA_LINKS.md`).

In [1]:
import os, sys, glob, inspect, time
import numpy as np
import pandas as pd

ROOT = os.path.abspath('..')                      # the project folder (this notebook is in notebooks/)
sys.path.insert(0, os.path.join(ROOT, 'experiments'))
sys.path.insert(0, os.path.join(ROOT, 'backend'))

from lib import data_prep                         # experiments/lib/data_prep.py: the cleaning
from amr_constants import ANTIBIOTIC_ALIASES, DRUG_CLASS_MAP, PHENOTYPE_MAP   # backend/amr_constants.py

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 70)

def show(fn):
    """Print a function's source code from the project."""
    print(inspect.getsource(fn))

def show_part(fn, start, end):
    """Print the lines of a function from the line containing `start` up to the one before `end`."""
    lines = inspect.getsource(fn).splitlines()
    i = next(n for n, l in enumerate(lines) if start in l)
    j = next(n for n, l in enumerate(lines) if n > i and end in l)
    print('\n'.join(lines[i:j]))

print('project:', ROOT)

project: C:\Users\HP\OneDrive - Higher Education Commission\Desktop\Final Year Project\FYP2\FYP1


## 1. The raw data

**Source:** the BV-BRC database (`bv-brc.org/api/genome_amr/`), one record per (genome, antibiotic, test). The models were trained on the **complete export pinned on 2026-09-28: 17,585,506 records** (`scripts/bvbrc_download/download_amr_full.py`).

That export is 4.9 GB and is kept on the team's Google Drive. This laptop holds a newer download of the **laboratory records only** (2026-11-08), which is used here to show the raw rows and each cleaning step. BV-BRC keeps adding records, so this newer download is larger than the laboratory part of the pinned export (section 9 explains the difference).

Only 10 of the 24 columns are read (`data_prep.RAW_COLUMNS`); the **Genome ID is read as text**, because read as a number, `195.304` and `195.3040` become the same genome.

In [2]:
files = sorted(glob.glob(os.path.join(data_prep.data_root(), 'amr_full', 'lab', '**', '*.csv'), recursive=True))
t0 = time.time()
raw = pd.concat([pd.read_csv(f, low_memory=False, dtype=data_prep.GENOME_ID_TEXT,
                             usecols=lambda c: c in data_prep.RAW_COLUMNS) for f in files], ignore_index=True)
print(f'{len(files)} files, {len(raw):,} raw records, columns read: {data_prep.RAW_COLUMNS} ({time.time() - t0:.0f} s)')
raw.head(6)

16 files, 1,537,259 raw records, columns read: ['Taxon ID', 'Genome ID', 'Genome Name', 'Antibiotic', 'Resistant Phenotype', 'Measurement', 'Measurement Value', 'Measurement Unit', 'Evidence', 'Computational Method Performance'] (24 s)


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Value,Measurement Unit,Computational Method Performance,Evidence
0,562.0,562.144628,Escherichia coli 10,ceftriaxone,NaN,<=0.25,0.25,mg/L,NaN,Laboratory Method
1,562.0,562.182452,Escherichia coli D17,amikacin,Susceptible,<=4,4,mg/L,NaN,Laboratory Method
2,1773.0,1773.31197,Mycobacterium tuberculosis 05/PTAN-0162/TAN-479/1,amikacin,NaN,0.25,0.25,mg/L,NaN,Laboratory Method
3,1352.0,1352.23965,Enterococcus faecium P-4545,levofloxacin,NaN,8,8,mg/L,NaN,Laboratory Method
4,562.0,562.5691,Escherichia coli CVM N36113PS,trimethoprim/sulfamethoxazole,NaN,<=0.12,0.12,mg/L,NaN,Laboratory Method
5,573.0,573.105958,Klebsiella pneumoniae Q203,cefalotin,Resistant,NaN,NaN,NaN,NaN,Laboratory Method


## 2. Step 1: keep records with a resistance call, and turn the call into a label

Many records are measurements BV-BRC never turned into a call; they have no label and are dropped. The calls become a binary target: **Resistant, Intermediate, Nonsusceptible and Reduced Susceptibility → 1**, Susceptible and Susceptible-dose dependent → 0. Calling an intermediate isolate resistant is the safer error.

In [3]:
print('label map (backend/amr_constants.py):', PHENOTYPE_MAP)
print()
print(raw['Resistant Phenotype'].fillna('(empty)').value_counts().to_string())
usable = raw[raw['Resistant Phenotype'].isin(PHENOTYPE_MAP)].copy()
print(f'\nkept: {len(usable):,} of {len(raw):,} records have a usable call ({len(usable) / len(raw):.1%})')

label map (backend/amr_constants.py): {'Susceptible': 0, 'Susceptible-dose dependent': 0, 'Resistant': 1, 'Intermediate': 1, 'Nonsusceptible': 1, 'Reduced Susceptibility': 1}



Resistant Phenotype
(empty)                       665684
Susceptible                   560939
Resistant                     282010
Intermediate                   25138
Nonsusceptible                  3355
Reduced Susceptibility           116
Ri                                 9
Susceptible-dose dependent         7
IR                                 1



kept: 871,565 of 1,537,259 records have a usable call (56.7%)


## 3. Step 2: the MIC (minimum inhibitory concentration)

The measurement is split into a **sign** (`<=`, `>`, `=`, ...) and a **value**, and `log(1 + MIC)` is added for the model. Disk-diffusion results are zone diameters in **mm**, not concentrations: a 25 mm zone is not an MIC of 25 mg/L, so those rows keep their label but get no MIC (the fix of cleaning v7). The code:

In [4]:
show_part(data_prep.clean, '# ── MIC', '# ── Organism')

    # ── MIC: sign and magnitude ──────────────────────────────────────────
    meas = df['Measurement'].astype('string')
    df['mic_sign'] = meas.str.extract(MIC_SIGN_RE, expand=False).fillna('unknown')
    mic = meas.str.extract(r'(\d+\.?\d*)', expand=False).astype('Float64')
    fallback = df['Measurement Value'].astype('string').str.extract(
        r'(\d+\.?\d*)', expand=False).astype('Float64')
    df['mic_value'] = mic.fillna(fallback).astype('float64')
    if version_number(version) >= 7 and 'Measurement Unit' in df:
        # Disk diffusion reports a zone diameter in mm, not an MIC
        zone = df['Measurement Unit'].astype('string').str.strip().str.lower().eq('mm').fillna(False)
        df.loc[zone.to_numpy(), 'mic_value'] = np.nan
        df.loc[zone.to_numpy(), 'mic_sign'] = 'unknown'
        if verbose:
            print(f'[clean] {int(zone.sum()):,} rows measured in mm (disk diffusion): no MIC')
    df['has_mic'] = df['mic_value'].notna().astype(int)
    df['mic_log'] 

In [5]:
meas = usable['Measurement'].astype('string')
demo = pd.DataFrame({'Measurement': meas, 'Unit': usable['Measurement Unit']})
demo['mic_sign'] = meas.str.extract(data_prep.MIC_SIGN_RE, expand=False).fillna('unknown')
demo['mic_value'] = meas.str.extract(r'(\d+\.?\d*)', expand=False).astype('float64')
zone = demo['Unit'].astype('string').str.strip().str.lower().eq('mm').fillna(False).to_numpy()
demo.loc[zone, 'mic_value'] = np.nan
demo.loc[zone, 'mic_sign'] = 'unknown'
print(f'rows measured in mm (disk diffusion), given no MIC: {zone.sum():,}')
pd.concat([demo[~zone].dropna(subset=['Measurement']).drop_duplicates('mic_sign').head(5),
           demo[zone].head(2)])

rows measured in mm (disk diffusion), given no MIC: 8,380


,Measurement,Unit,mic_sign,mic_value
1,<=4,mg/L,<=,4.00
6,>8,mg/L,>,8.00
8,0.06,mg/L,unknown,0.06
36,>=64,mg/L,>=,64.00
85,=2,mg/L,=,2.00
131,6,mm,unknown,NaN
206,30,mm,unknown,NaN


## 4. Step 3: the organism

Genus and species come from the genome name. BV-BRC's taxon IDs are mostly **strain** level (*E. coli* alone has 2,423 of them), so each is also mapped to its **species**-level taxon from NCBI (`backend/taxon_species.csv`, built by `experiments/build_taxonomy.py`).

In [6]:
name = usable['Genome Name'].astype('string').str.strip().str.strip('"\'')
parts = name.str.split(n=2, expand=True)
org = pd.DataFrame({'Genome Name': usable['Genome Name'], 'genus': parts[0].str.capitalize(),
                    'species': parts[1].str.lower(), 'Taxon ID (strain)': pd.to_numeric(usable['Taxon ID'], errors='coerce')})
org = org.drop_duplicates('genus').head(6).copy()
org['species-level taxon'] = data_prep.species_taxon_ids(org['Taxon ID (strain)'].fillna(0).astype('int64')).to_numpy()
org

,Genome Name,genus,species,Taxon ID (strain),species-level taxon
1,Escherichia coli D17,Escherichia,coli,562.0,562
5,Klebsiella pneumoniae Q203,Klebsiella,pneumoniae,573.0,573
8,Mycobacterium tuberculosis 705999,Mycobacterium,tuberculosis,1773.0,1773
10,Streptococcus pneumoniae strain GPSC4,Streptococcus,pneumoniae,1313.0,1313
12,Staphylococcus aureus J581,Staphylococcus,aureus,1280.0,1280
15,Acinetobacter baumannii strain MRSN20315,Acinetobacter,baumannii,470.0,470


## 5. Step 4: antibiotic names and drug classes

Names are lower-cased and trimmed, then the alias map in `backend/amr_constants.py` renames spelling variants and drops entries that are not drugs (drug classes such as `carbapenem`, or `instrument`). Each drug is then mapped to its class.

In [7]:
ab = usable['Antibiotic'].astype('string').str.strip().str.lower()
renames = {k: v for k, v in ANTIBIOTIC_ALIASES.items() if v is not None}
drops = {k for k, v in ANTIBIOTIC_ALIASES.items() if v is None}
print(f'alias map: {len(renames)} spellings renamed, {len(drops)} non-drugs dropped')
print(f'in these records: {ab.nunique()} raw names; {ab.isin(renames).sum():,} rows renamed, {ab.isin(drops).sum():,} rows dropped')
print(f'drug classes in the map: {len(set(DRUG_CLASS_MAP.values()))} (the cleaned training table uses 23 of them, section 9)')
pd.DataFrame({'raw name': list(renames)[:8], 'becomes': [renames[k] for k in list(renames)[:8]],
              'drug class': [DRUG_CLASS_MAP.get(renames[k], 'other') for k in list(renames)[:8]]})

alias map: 40 spellings renamed, 10 non-drugs dropped


in these records: 178 raw names; 35,415 rows renamed, 9,023 rows dropped
drug classes in the map: 24 (the cleaned training table uses 23 of them, section 9)


,raw name,becomes,drug class
0,ampicillin-sulbactam,ampicillin/sulbactam,beta_lactam
1,ampicillin_clavulanic_acid,amoxicillin/clavulanic acid,beta_lactam
2,amoxicillin-clavulanic acid,amoxicillin/clavulanic acid,beta_lactam
3,piperacillin-tazobactam,piperacillin/tazobactam,beta_lactam
4,trimethoprim-sulfamethoxazole,trimethoprim/sulfamethoxazole,sulfonamide
5,sulfamethoxazole/trimethoprim,trimethoprim/sulfamethoxazole,sulfonamide
6,co_trimoxazole,trimethoprim/sulfamethoxazole,sulfonamide
7,co-trimoxazole,trimethoprim/sulfamethoxazole,sulfonamide


## 6. Step 5: where each label comes from

BV-BRC mixes **laboratory** results with **computational** predictions (labels predicted from the genome by BV-BRC's own models). Every row keeps a `label_source`, so a model can be trained or tested on laboratory rows only. That is why the genome models are quoted on laboratory rows: computational labels were themselves predicted from the genome, so scoring a genome model on them is partly circular.

The newer download on this laptop holds laboratory records only, so every row below is "Laboratory Method". In the pinned export the models were trained on, 649,944 of the 7,847,110 cleaned rows (8.3%) are laboratory results and the rest computational (section 9).

In [8]:
print(usable['Evidence'].fillna('(empty)').value_counts().to_string())

Evidence
Laboratory Method    871565


## 7. Step 6: one row per genome and antibiotic

A genome tested several times against the same drug would otherwise appear several times, and could land on both sides of a train/test split. One row is kept per pair, a laboratory result first.

In [9]:
ab_clean = ab[~ab.isin(drops)].replace(renames)            # after the name step, as clean() does
key = usable.loc[ab_clean.index, 'Genome ID'].astype(str) + ' | ' + ab_clean
dup = key.duplicated(keep=False)
print(f'{dup.sum():,} records share a (genome, antibiotic) pair with another; {key.duplicated().sum():,} are removed')
example = key[dup].iloc[0]
usable.loc[key[key == example].index, ['Genome ID', 'Antibiotic', 'Resistant Phenotype', 'Measurement', 'Evidence']]

88,549 records share a (genome, antibiotic) pair with another; 46,693 are removed


,Genome ID,Antibiotic,Resistant Phenotype,Measurement,Evidence
15,470.1556,gentamicin,Resistant,>8,Laboratory Method
1135417,470.1556,gentamicin,Resistant,>=16,Laboratory Method
1458531,470.1556,gentamicin,Resistant,>8,Laboratory Method


## 8. The whole cleaning, in one call

`data_prep.clean()` does all of the above, vectorised. This is the exact function that built the training table:

In [10]:
show(data_prep.clean)

def clean(df_raw, normalize_antibiotics=True, verbose=True, version=CLEAN_VERSION):
    """Raw export → modelling table. Vectorised; no per-row apply.

    `version` switches on the fixes of that cleaning version and later, so an
    older version's table can still be rebuilt exactly.
    """
    df = df_raw.copy()

    # ── MIC: sign and magnitude ──────────────────────────────────────────
    meas = df['Measurement'].astype('string')
    df['mic_sign'] = meas.str.extract(MIC_SIGN_RE, expand=False).fillna('unknown')
    mic = meas.str.extract(r'(\d+\.?\d*)', expand=False).astype('Float64')
    fallback = df['Measurement Value'].astype('string').str.extract(
        r'(\d+\.?\d*)', expand=False).astype('Float64')
    df['mic_value'] = mic.fillna(fallback).astype('float64')
    if version_number(version) >= 7 and 'Measurement Unit' in df:
        # Disk diffusion reports a zone diameter in mm, not an MIC
        zone = df['Measurement Unit'].astype('string').str.strip().str.lower().eq('

In [11]:
t0 = time.time()
cleaned_new = data_prep.clean(usable)
print(f'({time.time() - t0:.0f} s)')
cleaned_new.head()

[clean] 8,380 rows measured in mm (disk diffusion): no MIC


[clean] 815,849 rows (46,693 duplicates dropped)
[clean] susceptible=526,003 resistant=289,846 (35.5% resistant)


[clean] genomes=101,331 drugs=158 genera=43
(31 s)


,Genome ID,Taxon ID,Antibiotic,drug_class,genus,species,mic_sign,mic_value,mic_log,has_mic,is_lab_confirmed,computational_f1,label_source,target,species_taxon_id
0,106654.148,106654,cefepime,beta_lactam,Acinetobacter,nosocomialis,unknown,64.0,4.174387,1,1,1.0,lab,1,106654
1,106654.148,106654,ceftazidime,beta_lactam,Acinetobacter,nosocomialis,unknown,16.0,2.833213,1,1,1.0,lab,1,106654
2,106654.148,106654,ceftriaxone,beta_lactam,Acinetobacter,nosocomialis,unknown,16.0,2.833213,1,1,1.0,lab,1,106654
3,106654.148,106654,ciprofloxacin,fluoroquinolone,Acinetobacter,nosocomialis,unknown,2.0,1.098612,1,1,1.0,lab,1,106654
4,106654.148,106654,gentamicin,aminoglycoside,Acinetobacter,nosocomialis,unknown,16.0,2.833213,1,1,1.0,lab,1,106654


## 9. The table the models were trained on

The models were trained on the pinned export (2026-09-28), cleaned by the same function and saved as `experiments/cache/clean_v7_amr_full_norm.pkl`. Below, every figure of the Data chapter (section 5) is **recomputed from that table** and compared with the reported value.

*Why not rebuild it from raw here?* The pinned 4.9 GB export is on Google Drive, not on this laptop. The newer laboratory download above, cleaned by the same code, gives more laboratory rows than the pinned export, because BV-BRC added records between September and November. The models and every reported number use the pinned export.

In [12]:
t0 = time.time()
df = data_prep.get_clean(verbose=False)            # the cached v7 table of the pinned export
lab = df[df['label_source'] == 'lab']
print(f'loaded {len(df):,} rows x {df.shape[1]} columns in {time.time() - t0:.0f} s')

def figures(d):
    return {'rows': len(d), 'genomes': d['Genome ID'].nunique(),
            'resistant %': round(100 * d['target'].mean(), 1), 'with an MIC %': round(100 * d['has_mic'].mean(), 1)}

reported = {'all rows':        {'rows': 7_847_110, 'genomes': 439_542, 'resistant %': 28.0, 'with an MIC %': 2.7},
            'laboratory rows': {'rows': 649_944,   'genomes': 87_325,  'resistant %': 33.7, 'with an MIC %': 33.1}}
measured = {'all rows': figures(df), 'laboratory rows': figures(lab)}
fmt = lambda v: f'{v:,}' if isinstance(v, (int, np.integer)) else f'{v:.1f}'
check = pd.DataFrame([{'table': t, 'figure': k, 'reported': fmt(reported[t][k]), 'recomputed': fmt(measured[t][k]),
                       'match': '✓' if reported[t][k] == measured[t][k] else '✗'}
                      for t in reported for k in reported[t]])
other = {'antibiotics': (126, df['Antibiotic'].nunique()), 'drug classes': (23, df['drug_class'].nunique()),
         'genera': (40, df['genus'].nunique()), 'species-level taxa': (247, df['species_taxon_id'].nunique()),
         'rows per genome (median)': (20, int(df.groupby('Genome ID').size().median()))}
check = pd.concat([check, pd.DataFrame([{'table': 'all rows', 'figure': k, 'reported': fmt(r), 'recomputed': fmt(m),
                                         'match': '✓' if r == m else '✗'} for k, (r, m) in other.items()])],
                  ignore_index=True)
check

loaded 7,847,110 rows x 15 columns in 5 s


,table,figure,reported,recomputed,match
0,all rows,rows,"7,847,110","7,847,110",✓
1,all rows,genomes,"439,542","439,542",✓
2,all rows,resistant %,28.0,28.0,✓
3,all rows,with an MIC %,2.7,2.7,✓
4,laboratory rows,rows,"649,944","649,944",✓
5,laboratory rows,genomes,"87,325","87,325",✓
6,laboratory rows,resistant %,33.7,33.7,✓
7,laboratory rows,with an MIC %,33.1,33.1,✓
8,all rows,antibiotics,126,126,✓
9,all rows,drug classes,23,23,✓


In [13]:
assert (check['match'] == '✓').all(), 'a recomputed figure differs from the report'
print('Every figure matches docs/report/Chapter_Data.md, section 5.')
print()
print('Largest genera (rows):')
print(df['genus'].value_counts().head(5).to_string())

Every figure matches docs/report/Chapter_Data.md, section 5.

Largest genera (rows):


genus
Escherichia       3562956
Salmonella        1083062
Mycobacterium      979138
Klebsiella         703539
Staphylococcus     366887


## 10. From the cleaned table to each model's inputs

| Model | What it takes from the cleaned data | Where the code is shown in full |
|---|---|---|
| Forecaster (`/forecast`) | 14 features per row: antibiotic, drug class, genus, species, species-level taxon ID, the MIC (value, sign, log, whether there is one), whether the label is from a lab and the computational method's F1, and three out-of-fold resistance rates (drug; organism and drug; genus and drug) | `01_forecaster_model.ipynb` |
| Gene model (`/predict`) | AMRFinderPlus resistance genes and mutations of the genome, plus antibiotic and drug class | `02_genome_models.ipynb` |
| k-mer model (`/predict`) | 256 4-mer frequencies of the complete genome, GC content, genome length, antibiotic | `02_genome_models.ipynb`, and below |
| Timeline ML model (`/timeline`) | Laboratory rows with a collection year, grouped into a yearly resistant share per genus and antibiotic | `04_timeline_ml_and_gan.ipynb` |
| RL agent (`/timeline`) | No table: it learns in the simulation | `05_rl_agent.ipynb` |

The forecaster's 14 inputs, read from the served model file, and its preprocessing choices from its configuration (species-level taxa, a split by genome with 20% of genomes for testing, out-of-fold resistance rates):

In [14]:
import json, lightgbm as lgb
booster = lgb.Booster(model_file=os.path.join(ROOT, 'backend', 'trained_models', 'amr_lgbm_final_model.txt'))
print(len(booster.feature_name()), 'features:', booster.feature_name())
cfg = json.load(open(os.path.join(ROOT, 'experiments', 'configs', 'D3_forecaster_deploy_v7.json')))
print(json.dumps({k: cfg[k] for k in ['data', 'split', 'features']}, indent=2))

14 features: ['Taxon_ID', 'Antibiotic', 'drug_class', 'genus', 'species', 'mic_sign', 'is_lab_confirmed', 'computational_f1', 'mic_value', 'mic_log', 'has_mic', 'ab_resistance_rate', 'taxon_ab_resistance_rate', 'genus_ab_resistance_rate']
{
  "data": {
    "taxon_level": "species",
    "source": "amr_full"
  },
  "split": {
    "strategy": "grouped",
    "test_size": 0.2,
    "seed": 42
  },
  "features": {
    "target_encoding": "oof"
  }
}


**A genome becomes model inputs.** The genome below, a *Salmonella enterica* assembly (BV-BRC 108619.160), is in the **test** set of both genome models, so no model trained on it. The k-mer model's preprocessing counts every 4-letter DNA word inside each contig (A/C/G/T windows only) and turns the counts into frequencies, with the GC content and length:

In [15]:
from ml_models.genome_predictor import GenomeModelPredictor, genome_counts
fasta = open(os.path.join(ROOT, 'Data', 'genomes_full', '108619.160.fna')).read()
contigs = fasta.count('>')
kmer_model = GenomeModelPredictor(os.path.join(ROOT, 'backend', 'trained_models'), folder='genome')
X, length = kmer_model.features_frame(fasta, 'ciprofloxacin', 'Salmonella', 'enterica')
kmers = [c for c in X.columns if c.startswith('k4_')]
print(f'{contigs} contig(s), {length:,} bp -> {X.shape[1]} features: {len(kmers)} 4-mer frequencies + {[c for c in X.columns if c not in kmers]}')
X[kmers].T.sort_values(by=X.index[0], ascending=False).head(8).T

[Genome] Loaded G_kmer_deploy from genome/.


46 contig(s), 4,909,643 bp -> 259 features: 256 4-mer frequencies + ['Antibiotic', 'genome_gc', 'genome_length_mb']


,k4_GCGC,k4_GGCG,k4_CGCC,k4_AAAA,k4_CGGC,k4_GCCG,k4_CGCG,k4_TTTT
0,0.010458,0.010048,0.009734,0.008122,0.008109,0.008015,0.007973,0.007961
